# 2.5 — Tracking a Nonstationary Problem

---

## When averaging is the wrong thing to do

Sample averaging is *optimal* when $q_*$ is constant. Almost no interesting problem has
constant $q_*$. In reinforcement learning proper, the environment is nonstationary
almost by construction: as your policy improves, the value of an action changes.

So we want an estimator that keeps *forgetting*. Replace $1/n$ with a constant $\alpha$:

$$Q_{n+1} \;\doteq\; Q_n + \alpha\big[R_n - Q_n\big], \qquad \alpha \in (0, 1]$$

## Unrolling it: exponential recency-weighted average

$$
\begin{aligned}
Q_{n+1} &= Q_n + \alpha[R_n - Q_n] \\
&= \alpha R_n + (1-\alpha) Q_n \\
&= \alpha R_n + (1-\alpha)\big[\alpha R_{n-1} + (1-\alpha) Q_{n-1}\big] \\
&= \alpha R_n + (1-\alpha)\alpha R_{n-1} + (1-\alpha)^2 Q_{n-1} \\
&\;\;\vdots \\
&= \boxed{\;(1-\alpha)^n Q_1 \;+\; \sum_{i=1}^{n} \alpha(1-\alpha)^{n-i} R_i\;}
\end{aligned}
$$

Two things to read off this:

1. **The weights sum to one:** $(1-\alpha)^n + \sum_{i=1}^{n}\alpha(1-\alpha)^{n-i} = 1$.
   So it is a genuine weighted average.
2. **The weight on $R_i$ decays exponentially in how old it is.** The half-life is
   $\log(0.5)/\log(1-\alpha)$ steps; the effective window is about $1/\alpha$ samples.

The $(1-\alpha)^n Q_1$ term never fully vanishes — a constant-$\alpha$ estimator has a
permanent, exponentially-fading memory of where it started. Remember this for 2.6.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from bandit_utils import (Testbed, run_bandit, plot_pair, argmax_random_tiebreak,
                          hide, banner, RUNS, STEPS)

plt.rcParams["figure.dpi"] = 110
banner()

In [ ]:
class EpsGreedy:
    def __init__(self, k, runs, rng, eps=0.1, Q_init=0.0, alpha=None):
        self.k, self.runs, self.rng, self.eps = k, runs, rng, eps
        self.alpha = alpha
        self.Q = np.full((runs, k), float(Q_init))
        self.N = np.zeros((runs, k))

    def act(self):
        greedy = argmax_random_tiebreak(self.Q, self.rng)
        rand = self.rng.integers(0, self.k, size=self.runs)
        explore = self.rng.random(self.runs) < self.eps
        return np.where(explore, rand, greedy)

    def update(self, a, r):
        idx = np.arange(self.runs)
        self.N[idx, a] += 1
        step = self.alpha if self.alpha else 1.0 / self.N[idx, a]
        self.Q[idx, a] += step * (r - self.Q[idx, a])


def eps_agent(eps, Q_init=0.0, alpha=None):
    return lambda k, runs, rng: EpsGreedy(k, runs, rng, eps=eps,
                                          Q_init=Q_init, alpha=alpha)

print("agent defined")

## Half-life: how far back does the estimate see?

$$\text{half-life} = \frac{\ln 0.5}{\ln(1-\alpha)} \qquad\qquad
\text{effective sample size} \approx \frac{2-\alpha}{\alpha} \approx \frac{1}{\alpha}\;\;(\alpha \text{ small})$$

In [ ]:
alphas = np.array([0.01, 0.05, 0.1, 0.2, 0.5])
half_life = np.log(0.5) / np.log(1 - alphas)
ess = (2 - alphas) / alphas

print(f"{'alpha':>7} {'half-life':>11} {'eff. samples':>14}")
for a, h, e in zip(alphas, half_life, ess):
    print(f"{a:>7} {h:>11.1f} {e:>14.1f}")

fig, ax = plt.subplots(figsize=(9, 4))
age = np.arange(0, 120)
for a in alphas:
    ax.plot(age, a * (1 - a) ** age, label=f"$\\alpha$={a}")
ax.set_xlabel("Age of the reward (steps ago)")
ax.set_ylabel("Weight in current estimate")
ax.set_title("Exponential recency weighting")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

---

### Predict first

We are about to make the testbed nonstationary: all ten $q_*(a)$ start
**equal** and then each takes an independent random walk, adding
$\mathcal{N}(0, 0.01^2)$ every step. This is Exercise 2.5.

1. Which does better over 10,000 steps: $\alpha = 1/n$ (sample average) or $\alpha = 0.1$?
2. Will the sample-average method's % optimal curve keep **rising**, **flatten off**, or
   actually **decline** over time? Where roughly does it end up after 10,000 steps?
3. Why do the $q_*$ values all start equal here — what would be different if they were
   drawn from $\mathcal{N}(0,1)$ as usual?

*Write your guess down (mentally or in the cell below) before running the next cell. The point is not to be right — it is to make the surprise informative when you are wrong.*

In [ ]:
STEPS_NS = 10000
kw = {"drift_sigma": 0.01, "equal_start": True}

sample_avg = run_bandit(eps_agent(0.1), testbed_kwargs=kw, steps=STEPS_NS, seed=1)
const_a = run_bandit(eps_agent(0.1, alpha=0.1), testbed_kwargs=kw, steps=STEPS_NS, seed=1)

fig, axes = plot_pair([sample_avg, const_a],
                      ["sample average  $\\alpha_n = 1/n$",
                       "constant  $\\alpha = 0.1$"],
                      title="Exercise 2.5 - nonstationary testbed, random-walk $q_*$")
plt.show()

for lab, r in [("1/n", sample_avg), ("alpha=0.1", const_a)]:
    print(f"{lab:<10} avg reward (all)={r['rewards'].mean():.3f}  "
          f"(last 1000)={r['rewards'][-1000:].mean():.3f}  "
          f"%opt(last 1000)={r['optimal'][-1000:].mean():.1f}")

In [ ]:
hide('''<b>1.</b> Constant &alpha; wins, and the gap widens without bound. Its estimate
tracks the moving q*; the sample average is dragged down by ancient, now-irrelevant data.
<br><br><b>2.</b> It rises and then <b>flattens off</b>, stalling around 45% while constant
&alpha; climbs past 75% by step 10,000 and keeps going. (A natural guess is that it
<i>declines</i> &mdash; it does not, even out to 40,000 steps. Check the printout.) The
mechanism: as n grows, &alpha;_n = 1/n shrinks, so the estimator becomes more and more inert
exactly as the target drifts further from where it was measured. It does not forget how to
rank arms; it just stops being able to keep up. The failure is a <i>ceiling</i>, not a collapse.
<br><br><b>3.</b> Starting all q* equal makes the problem purely about tracking. If they
started spread out from N(0,1), the initial 1-in-10 identification problem would dominate
the early curve and muddy the comparison &mdash; the drift needs thousands of steps to
matter, by which time both methods have already solved the easy part. Equal starts make
the drift the <i>only</i> thing that matters.''')

## What the estimates are actually doing

The averaged curves show *that* sample-averaging fails. This shows *why*: watch a single
arm's true value wander and both estimators try to follow it.

In [ ]:
rng = np.random.default_rng(4)
T = 3000
q_true = np.cumsum(rng.normal(0, 0.03, size=T))
rewards = rng.normal(q_true, 1.0)

Q_sa, Q_c1, Q_c2 = np.zeros(T), np.zeros(T), np.zeros(T)
qa = qb = qc = 0.0
for t in range(T):
    qa += (rewards[t] - qa) / (t + 1)
    qb += 0.1 * (rewards[t] - qb)
    qc += 0.01 * (rewards[t] - qc)
    Q_sa[t], Q_c1[t], Q_c2[t] = qa, qb, qc

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(q_true, color="k", lw=2.5, label="true $q_*(a)$ (random walk)")
axes[0].plot(Q_sa, label="$\\alpha_n=1/n$", lw=1.2)
axes[0].plot(Q_c2, label="$\\alpha=0.01$", lw=1.2)
axes[0].plot(Q_c1, label="$\\alpha=0.1$", lw=1.0, alpha=0.8)
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_ylabel("value")
axes[0].set_title("Tracking a drifting target")

for Q, lab in [(Q_sa, "1/n"), (Q_c2, "0.01"), (Q_c1, "0.1")]:
    axes[1].plot(np.abs(Q - q_true), label=f"|error| $\\alpha$={lab}", lw=1)
axes[1].set_xlabel("Step"); axes[1].set_ylabel("|Q - q*|")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

for Q, lab in [(Q_sa, "1/n"), (Q_c2, "alpha=0.01"), (Q_c1, "alpha=0.1")]:
    print(f"{lab:<12} mean |error| over last 1000 = {np.abs(Q-q_true)[-1000:].mean():.3f}")

## The bias-variance trade-off in $\alpha$

There is a genuine sweet spot, and it depends on the ratio of **drift speed** to
**reward noise**:

- $\alpha$ **too small** → estimate lags behind the drift. This is *bias* (tracking error).
- $\alpha$ **too large** → estimate jitters with every noisy sample. This is *variance*.

Roughly, mean squared error $\approx \underbrace{\frac{\sigma_{\text{drift}}^2}{\alpha^2}\cdot c}_{\text{lag}} + \underbrace{\frac{\alpha}{2-\alpha}\sigma_R^2}_{\text{noise}}$,
minimised at some interior $\alpha^*$ that grows with the drift rate.

In [ ]:
def tracking_mse(alpha, drift=0.03, noise=1.0, T=4000, seed=0):
    rng = np.random.default_rng(seed)
    q = np.cumsum(rng.normal(0, drift, size=T))
    r = rng.normal(q, noise)
    Q = 0.0; err = np.empty(T)
    for t in range(T):
        Q += alpha * (r[t] - Q)
        err[t] = (Q - q[t]) ** 2
    return err[T // 4:].mean()

alphas = np.logspace(-3, -0.1, 30)
fig, ax = plt.subplots(figsize=(9, 4.5))
for drift in [0.005, 0.02, 0.08]:
    mse = [np.mean([tracking_mse(a, drift=drift, seed=s) for s in range(12)])
           for a in alphas]
    ax.loglog(alphas, mse, "o-", ms=3, label=f"drift $\\sigma$={drift}")
    ax.scatter([alphas[int(np.argmin(mse))]], [min(mse)], s=90, zorder=5,
               facecolors="none", edgecolors="crimson", lw=2)
ax.set_xlabel(r"$\alpha$"); ax.set_ylabel("mean squared tracking error")
ax.set_title("Faster drift -> larger optimal step size (circles = minima)")
ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()

### Interactive: drift rate vs. step size

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets not installed - run:  pip install ipywidgets")

In [ ]:
def ns_experiment(alpha=0.1, drift=0.01, steps=4000):
    kw = {"drift_sigma": drift, "equal_start": True}
    a = run_bandit(eps_agent(0.1), testbed_kwargs=kw, steps=steps, runs=200, seed=1)
    b = run_bandit(eps_agent(0.1, alpha=alpha), testbed_kwargs=kw, steps=steps,
                   runs=200, seed=1)
    plot_pair([a, b], ["$1/n$", f"$\\alpha$={alpha}"],
              title=f"drift $\\sigma$={drift}")
    plt.show()

if HAVE_WIDGETS:
    W.interact(ns_experiment,
               alpha=W.FloatLogSlider(value=0.1, base=10, min=-3, max=-0.15, step=0.1),
               drift=W.FloatLogSlider(value=0.01, base=10, min=-3, max=-0.7, step=0.1),
               steps=W.fixed(4000))
else:
    ns_experiment(0.1, 0.01)

## Exercise 2.4 — weights under a *varying* step size

> If the step sizes $\alpha_n$ are not constant, what is the weight on each prior reward?

Unroll without assuming $\alpha$ is constant:

$$Q_{n+1} \;=\; \underbrace{\prod_{j=1}^{n}(1-\alpha_j)}_{\text{weight on } Q_1} Q_1
\;+\; \sum_{i=1}^{n} \alpha_i \left[\prod_{j=i+1}^{n} (1-\alpha_j)\right] R_i$$

Sanity check: with $\alpha_j = 1/j$ the bracket telescopes,
$\prod_{j=i+1}^{n}\frac{j-1}{j} = \frac{i}{n}$, giving weight
$\frac{1}{i}\cdot\frac{i}{n} = \frac{1}{n}$ on every $R_i$ — the flat weighting we saw in
2.4, and the weight on $Q_1$ is $\prod (1-1/j) = 0$ exactly. Verify numerically:

In [ ]:
def general_weights(alphas):
    n = len(alphas)
    w = np.zeros(n); w0 = 1.0
    for i, a in enumerate(alphas):
        w[:i] *= (1 - a); w0 *= (1 - a); w[i] = a
    return w0, w

for name, al in [("1/n", 1 / np.arange(1, 21)),
                 ("const 0.2", np.full(20, 0.2)),
                 ("decaying n^-0.6", np.arange(1, 21) ** -0.6)]:
    w0, w = general_weights(al)
    print(f"{name:<16} w(Q1)={w0:.4f}  w(R_1)={w[0]:.4f}  w(R_20)={w[-1]:.4f}  "
          f"total={w0 + w.sum():.6f}")

Every row totals exactly 1 — the update always produces a proper weighted average of
$Q_1$ and the observed rewards, whatever the step-size schedule. That is a useful
invariant to check when you write your own learning rules.

## Takeaways for 2.5

1. Constant $\alpha$ gives an exponential recency-weighted average with effective memory
   $\approx 1/\alpha$.
2. On drifting problems, $1/n$ does not merely learn slowly — its performance *degrades*
   over time, because $\alpha_n \to 0$ while the target keeps moving.
3. The optimal $\alpha$ trades lag (bias) against jitter (variance) and grows with the
   drift rate.
4. Constant $\alpha$ never forgets $Q_1$ entirely: the $(1-\alpha)^n Q_1$ term.

**Next:** turn that lingering $Q_1$ term into a feature — Section 2.6.